# K-SPADA Model Training

**Audience:** users training the liver cancer CHC-to-HCC reference model.  
**Goal:** reproduce the traceable historical-best hyperparameter setting without mixing prediction code into training.

A preserved slice-4 historical result reached an accuracy of **0.8330**. Therefore, this tutorial selects the matching traceable configuration rather than claiming a new exhaustive search: human GO biological-process pathways, zero-preserving bins 0–7, communication-graph `k=2`, spatial `k=5`, graph batch size 128, embedding dimension 48, two Transformer blocks, four attention heads, Adam with learning rate 0.001 and weight decay 5e-5, cosine final learning-rate ratio 0.01, and ten epochs.

In [ ]:
from pathlib import Path
import anndata as ad
import pandas as pd
import torch
import KSPADA

PROJECT_ROOT = Path('/home/lixiangyu/sh/program/K-SPADA')
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'liver_cancer_best'
PROJECT_NAME = 'liver_cancer_chc_to_hcc_best'

training_config = KSPADA.BEST_LIVER_CANCER_TRAINING
dataset_config = KSPADA.BEST_LIVER_CANCER_TUTORIAL
TRAIN_VALID_H5AD = dataset_config.train_valid_path

display(pd.Series(training_config.__dict__, name='value'))
display(pd.Series(training_config.to_training_kwargs(), name='training_argument'))
display(pd.Series(training_config.to_data_preparation_kwargs(), name='data_preparation_argument'))
display(pd.Series(dataset_config.as_dict(), name='value'))
TRAINING_DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Training device:', TRAINING_DEVICE)


## Step 1 — Validate the prepared training file

In [ ]:
if not TRAIN_VALID_H5AD.is_file():
    raise FileNotFoundError(TRAIN_VALID_H5AD)
KSPADA.validate_cgraph_fields(TRAIN_VALID_H5AD, mode='train')

training_adata = ad.read_h5ad(TRAIN_VALID_H5AD, backed='r')
actual_batches = set(training_adata.obs['batch'].astype(str).unique())
expected_batches = set(dataset_config.train_batches + dataset_config.validation_batches)
if actual_batches != expected_batches:
    raise ValueError(f'Batch mismatch: expected {sorted(expected_batches)}, found {sorted(actual_batches)}')
print('Training shape:', training_adata.shape)
print('Batches:', sorted(actual_batches))
training_adata.file.close()

## Step 2 — Train with the selected historical-best configuration

The remaining batch in the training file is automatically used for validation. `communication_neighbors` is intentionally absent from `to_training_kwargs()` because the communication graph has already been constructed during data preparation.

In [ ]:
MODEL_DIR = OUTPUT_DIR / PROJECT_NAME
model_dir = KSPADA.train_model(
    adata_input=TRAIN_VALID_H5AD,
    project=PROJECT_NAME,
    output_dir=OUTPUT_DIR,
    train_batches=dataset_config.train_batches,
    **training_config.to_training_kwargs(),
)
print('Model directory:', model_dir)


## Step 3 — Inspect model artifacts

In [ ]:
expected_artifacts = ['mask.npy', 'pathway.csv', 'label_dictionary.csv', 'model-9.pth']
artifact_table = pd.DataFrame({
    'artifact': expected_artifacts,
    'exists': [(MODEL_DIR / artifact).exists() for artifact in expected_artifacts],
    'path': [str(MODEL_DIR / artifact) for artifact in expected_artifacts],
})
display(artifact_table)

## Exercise

Change only `graph_batch_size` in a new `TrainingConfig` and keep every other parameter fixed for a controlled comparison.

In [ ]:
from dataclasses import replace
exercise_config = replace(training_config, graph_batch_size=64)
display(pd.Series(exercise_config.__dict__, name='value'))

## Pitfall and extension

**Pitfall:** the graph mini-batch size counts target cells; sampled neighbors make the actual subgraph larger.  
**Extension:** compare validation accuracy for graph batch sizes 64 and 128 while keeping the data split unchanged.